# Full VAE Pretraining (ChemBL + Zinc + Tox21)
This notebook mirrors the pretraining setup from `PreTrained_VAE.ipynb`, while incorporating optimization insights:
- lower learning rate (`5e-4`) with LR scheduler decay factor `0.25`
- KL annealing enabled
- free bits removed

Key dataset change:
- include `tox21_train` in the pretraining train split
- include `tox21_val` in the pretraining validation split
- enforce no canonical-smiles overlap across train/val/test (tox21 split assignment takes priority)

Training supports automatic crash recovery by resuming from the latest checkpoint when `AUTO_RESUME=True`.

### Imports and config

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import selfies as sf

try:
    import wandb
except ImportError:
    wandb = None

SEED = 42
MAX_LEN = 120  # max SELFIES token length before EOS
VAL_FRAC = 0.10
TEST_FRAC = 0.10

LATENT_DIM = 292
BATCH_SIZE = 128
LR = 5e-4
KL_ANNEAL_EPOCHS = 10
FREE_BITS_NATS = 0.0  # optimization insight: remove free bits

MIN_EPOCHS = 50
MAX_EPOCHS = 120
EARLY_STOPPING_PATIENCE = 12

LR_SCHEDULER_FACTOR = 0.25
LR_SCHEDULER_PATIENCE = 4

AUTO_RESUME = True

USE_WANDB = True
WANDB_PROJECT = "ai-for-toxicology"
WANDB_RUN_NAME = "full-pretrain-chembl-zinc-tox21-seqconv"

CHECKPOINT_DIR = Path("artifacts") / "full_pretraining_checkpoints"
CHECKPOINT_STEM = "paper_like_selfies_full_pretrain_seqconv_ce"
SAVE_EPOCH_CHECKPOINTS = True  # Set False if disk usage becomes too high.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print("device:", device)
print("torch:", torch.__version__)
print("selfies:", sf.__version__)
print("wandb:", "available" if wandb is not None else "not installed (optional)")

### Load ChemBL, Zinc, and Tox21 datasets

In [ ]:
DATA_ROOT = Path("data")
CHEMBL_PATH = DATA_ROOT / "Train" / "chembl_clean.csv"
ZINC_PATH = DATA_ROOT / "Train" / "zinc250k_clean.csv"
TOX21_TRAIN_PATH = DATA_ROOT / "Train" / "tox21_train_clean.csv"
TOX21_VAL_PATH = DATA_ROOT / "Val" / "tox21_val_clean.csv"
TOX21_TEST_PATH = DATA_ROOT / "Test" / "tox21_test_clean.csv"

for p in [CHEMBL_PATH, ZINC_PATH, TOX21_TRAIN_PATH, TOX21_VAL_PATH, TOX21_TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")


def load_smiles(path: Path) -> list[str]:
    df = pd.read_csv(path)
    if "canonical_smiles" not in df.columns:
        raise ValueError(f"{path} does not contain canonical_smiles")
    smiles = df["canonical_smiles"].dropna().astype(str).tolist()
    return list(dict.fromkeys(smiles))


chembl_smiles = load_smiles(CHEMBL_PATH)
zinc_smiles = load_smiles(ZINC_PATH)
tox21_train_smiles = load_smiles(TOX21_TRAIN_PATH)
tox21_val_smiles = load_smiles(TOX21_VAL_PATH)

pretrain_smiles = list(dict.fromkeys(chembl_smiles + zinc_smiles))

print(f"ChemBL unique SMILES:      {len(chembl_smiles):,}")
print(f"Zinc unique SMILES:        {len(zinc_smiles):,}")
print(f"Base pretraining unique:   {len(pretrain_smiles):,}")
print(f"Tox21 train unique SMILES: {len(tox21_train_smiles):,}")
print(f"Tox21 val unique SMILES:   {len(tox21_val_smiles):,}")

### Create base split, merge Tox21 train/val, enforce split isolation, and convert to SELFIES

In [ ]:
def smiles_to_selfies(smiles_list: list[str]) -> tuple[list[str], int]:
    out = []
    failed = 0
    for smi in smiles_list:
        try:
            out.append(sf.encoder(smi))
        except Exception:
            failed += 1
    return out, failed


def filter_selfies_len(selfies_list: list[str], max_len: int = MAX_LEN) -> list[str]:
    return [s for s in selfies_list if len(list(sf.split_selfies(s))) <= max_len]


def split_list(items: list[str], val_frac: float, test_frac: float, seed: int) -> tuple[list[str], list[str], list[str]]:
    rng = np.random.default_rng(seed)
    idx = np.arange(len(items))
    rng.shuffle(idx)

    n = len(items)
    n_test = max(1, int(round(n * test_frac)))
    n_val = max(1, int(round(n * val_frac)))

    if n_test + n_val >= n:
        n_test = max(1, n // 10)
        n_val = max(1, n // 10)

    test_idx = idx[:n_test]
    val_idx = idx[n_test:n_test + n_val]
    train_idx = idx[n_test + n_val:]

    train = [items[i] for i in train_idx]
    val = [items[i] for i in val_idx]
    test = [items[i] for i in test_idx]
    return train, val, test


# 1) Base ChemBL+Zinc split (same logic as the original notebook)
base_train_smiles, base_val_smiles, base_test_smiles = split_list(pretrain_smiles, VAL_FRAC, TEST_FRAC, seed=SEED)

# 2) Merge Tox21 while enforcing split isolation with tox21 priority
# Priority: tox21_train -> final train, tox21_val -> final val
# Remove any tox21 molecules from base splits to prevent leakage.
tox21_train_set = set(tox21_train_smiles)
tox21_val_set = set(tox21_val_smiles)

base_train_smiles = [s for s in base_train_smiles if s not in tox21_train_set and s not in tox21_val_set]
base_val_smiles = [s for s in base_val_smiles if s not in tox21_train_set and s not in tox21_val_set]
base_test_smiles = [s for s in base_test_smiles if s not in tox21_train_set and s not in tox21_val_set]

# In case of accidental overlap between tox21 train/val, keep train assignment priority.
tox21_val_smiles = [s for s in tox21_val_smiles if s not in tox21_train_set]

final_train_smiles = list(dict.fromkeys(base_train_smiles + tox21_train_smiles))
final_val_smiles = list(dict.fromkeys(base_val_smiles + tox21_val_smiles))
final_test_smiles = list(dict.fromkeys(base_test_smiles))

# 3) Integrity checks (canonical SMILES disjointness)
train_set = set(final_train_smiles)
val_set = set(final_val_smiles)
test_set = set(final_test_smiles)

assert train_set.isdisjoint(val_set), "train/val overlap detected"
assert train_set.isdisjoint(test_set), "train/test overlap detected"
assert val_set.isdisjoint(test_set), "val/test overlap detected"

print("Base split sizes before tox21 merge:")
print(f"  base train={len(base_train_smiles):,}, base val={len(base_val_smiles):,}, base test={len(base_test_smiles):,}")
print("Final split sizes after tox21 merge and overlap cleanup:")
print(f"  train={len(final_train_smiles):,}, val={len(final_val_smiles):,}, test={len(final_test_smiles):,}")

# 4) Convert each final split to SELFIES and filter length
train_selfies, train_failed = smiles_to_selfies(final_train_smiles)
val_selfies, val_failed = smiles_to_selfies(final_val_smiles)
test_selfies, test_failed = smiles_to_selfies(final_test_smiles)

train_before_filter = len(train_selfies)
val_before_filter = len(val_selfies)
test_before_filter = len(test_selfies)

train_selfies = filter_selfies_len(train_selfies, max_len=MAX_LEN)
val_selfies = filter_selfies_len(val_selfies, max_len=MAX_LEN)
test_selfies = filter_selfies_len(test_selfies, max_len=MAX_LEN)

print("SELFIES conversion failures by split:")
print(f"  train failed={train_failed:,}, val failed={val_failed:,}, test failed={test_failed:,}")
print("After SELFIES length filtering:")
print(f"  train={len(train_selfies):,}/{train_before_filter:,}, val={len(val_selfies):,}/{val_before_filter:,}, test={len(test_selfies):,}/{test_before_filter:,}")

### Tokenization and encoding

In [ ]:
PAD = "<PAD>"
UNK = "<UNK>"
EOS = "<EOS>"


def tokenize_selfies(s: str) -> list[str]:
    return list(sf.split_selfies(s))


train_tokens = [tokenize_selfies(s) for s in train_selfies]
vocab_tokens = sorted({tok for seq in train_tokens for tok in seq})
ALL_TOKENS = [PAD, UNK, EOS] + vocab_tokens
TOKEN_TO_IDX = {tok: i for i, tok in enumerate(ALL_TOKENS)}
IDX_TO_TOKEN = {i: tok for tok, i in TOKEN_TO_IDX.items()}

PAD_IDX = TOKEN_TO_IDX[PAD]
UNK_IDX = TOKEN_TO_IDX[UNK]
EOS_IDX = TOKEN_TO_IDX[EOS]

SEQ_LEN = MAX_LEN + 1
VOCAB_SIZE = len(ALL_TOKENS)

if SEQ_LEN < 29:
    raise ValueError("Sequence length is too small for conv kernels (9, 9, 11).")


def encode_selfies(s: str) -> list[int]:
    ids = [TOKEN_TO_IDX.get(tok, UNK_IDX) for tok in tokenize_selfies(s)]
    ids = ids[:MAX_LEN]
    ids.append(EOS_IDX)
    return ids


def encode_list(selfies_list: list[str]) -> np.ndarray:
    out = np.full((len(selfies_list), SEQ_LEN), PAD_IDX, dtype=np.int64)
    for i, s in enumerate(selfies_list):
        ids = encode_selfies(s)
        out[i, :len(ids)] = ids
    return out


train_x = encode_list(train_selfies)
val_x = encode_list(val_selfies)
test_x = encode_list(test_selfies)

print(f"train={train_x.shape}, val={val_x.shape}, test={test_x.shape}")
print(f"VOCAB_SIZE={VOCAB_SIZE}, SEQ_LEN={SEQ_LEN}")


### SELFIES VAE

In [ ]:
class TokenDataset(Dataset):
    def __init__(self, x: np.ndarray):
        self.x = torch.from_numpy(x).long()

    def __len__(self):
        return self.x.size(0)

    def __getitem__(self, idx):
        return self.x[idx]


class PaperLikeSelfiesVAE(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, latent_dim: int = 292):
        super().__init__()
        self.vocab_size = vocab_size
        self.seq_len = seq_len

        # Conv1d expects [batch, channels, length], so we encode one-hot SELFIES as [B, vocab, seq].
        self.conv_1 = nn.Conv1d(in_channels=vocab_size, out_channels=9, kernel_size=9)
        self.conv_2 = nn.Conv1d(in_channels=9, out_channels=9, kernel_size=9)
        self.conv_3 = nn.Conv1d(in_channels=9, out_channels=10, kernel_size=11)
        self.relu = nn.ReLU()

        with torch.no_grad():
            dummy = torch.zeros(1, vocab_size, seq_len)
            d = self.relu(self.conv_1(dummy))
            d = self.relu(self.conv_2(d))
            d = self.relu(self.conv_3(d))
            flat = d.flatten(1).size(1)

        self.linear_0 = nn.Linear(flat, 435)
        self.linear_1 = nn.Linear(435, latent_dim)
        self.linear_2 = nn.Linear(435, latent_dim)

        self.linear_3 = nn.Linear(latent_dim, 292)
        self.gru = nn.GRU(input_size=292, hidden_size=501, num_layers=3, batch_first=True)
        self.linear_4 = nn.Linear(501, vocab_size)

    def encoder(self, x_onehot: torch.Tensor):
        x = self.relu(self.conv_1(x_onehot))
        x = self.relu(self.conv_2(x))
        x = self.relu(self.conv_3(x))
        x = x.flatten(1)
        x = F.selu(self.linear_0(x))
        return self.linear_1(x), self.linear_2(x)

    def sampling(self, mean: torch.Tensor, logvar: torch.Tensor):
        eps = 1e-2 * torch.randn_like(logvar)
        return torch.exp(0.5 * logvar) * eps + mean

    def decode(self, z: torch.Tensor):
        z = F.selu(self.linear_3(z))
        z_seq = z.unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.gru(z_seq)
        logits = self.linear_4(out)
        return logits

    def forward(self, x_onehot: torch.Tensor):
        mean, logvar = self.encoder(x_onehot)
        z = self.sampling(mean, logvar)
        logits = self.decode(z)
        return logits, mean, logvar


def ids_to_onehot(x_ids: torch.Tensor, vocab_size: int):
    return F.one_hot(x_ids, num_classes=vocab_size).float().transpose(1, 2).contiguous()


def vae_loss(
    logits: torch.Tensor,
    x_ids: torch.Tensor,
    mean: torch.Tensor,
    logvar: torch.Tensor,
    *,
    pad_idx: int,
    beta: float = 1.0,
):
    vocab_size = logits.size(-1)
    recon_sum = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        x_ids.reshape(-1),
        ignore_index=pad_idx,
        reduction="sum",
    )

    kl_per_dim = -0.5 * (1 + logvar - mean.pow(2) - logvar.exp())
    kl_sum = kl_per_dim.sum()

    n_nonpad = (x_ids != pad_idx).sum().clamp(min=1)
    total = recon_sum + beta * kl_sum
    return total, recon_sum, kl_sum, n_nonpad


def kl_beta(epoch: int, anneal_epochs: int) -> float:
    if anneal_epochs <= 1:
        return 1.0
    return float(min(1.0, epoch / anneal_epochs))

### Training and evaluation helpers (scheduler, early stopping, checkpointing, auto-resume support)

In [ ]:
def make_loader(x: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(TokenDataset(x), batch_size=batch_size, shuffle=shuffle)


def init_wandb(*, run_name: str | None = None, epochs: int | None = None, start_epoch: int = 0):
    if not USE_WANDB:
        return None
    if wandb is None:
        raise ImportError("wandb is not installed. Install it or set USE_WANDB=False.")

    return wandb.init(
        project=WANDB_PROJECT,
        name=run_name or WANDB_RUN_NAME,
        config={
            "seed": SEED,
            "max_len": MAX_LEN,
            "latent_dim": LATENT_DIM,
            "min_epochs": MIN_EPOCHS,
            "max_epochs": epochs if epochs is not None else MAX_EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "kl_anneal_epochs": KL_ANNEAL_EPOCHS,
            "free_bits_nats": FREE_BITS_NATS,
            "lr_scheduler_factor": LR_SCHEDULER_FACTOR,
            "lr_scheduler_patience": LR_SCHEDULER_PATIENCE,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "start_epoch": start_epoch,
            "vocab_size": VOCAB_SIZE,
            "seq_len": SEQ_LEN,
        },
    )


def run_epoch(model: nn.Module, loader: DataLoader, *, optimizer=None, beta: float = 1.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_sum = 0.0
    recon_sum = 0.0
    kl_sum = 0.0
    n_samples = 0
    n_nonpad = 0
    n_correct = 0

    for x_ids in loader:
        x_ids = x_ids.to(device)
        x_onehot = ids_to_onehot(x_ids, VOCAB_SIZE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits, mean, logvar = model(x_onehot)
            total, recon, kl, nonpad = vae_loss(
                logits,
                x_ids,
                mean,
                logvar,
                pad_idx=PAD_IDX,
                beta=beta,
            )
            if is_train:
                total.backward()
                optimizer.step()

        mask = x_ids != PAD_IDX
        preds = logits.argmax(dim=-1)

        total_sum += total.item()
        recon_sum += recon.item()
        kl_sum += kl.item()
        n_samples += x_ids.size(0)
        n_nonpad += int(nonpad.item())
        n_correct += ((preds == x_ids) & mask).sum().item()

    return {
        "total": total_sum / max(n_samples, 1),
        "recon_per_token": recon_sum / max(n_nonpad, 1),
        "kl": kl_sum / max(n_samples, 1),
        "token_acc": n_correct / max(n_nonpad, 1),
    }


def evaluate(model: nn.Module, x: np.ndarray, *, beta: float):
    loader = make_loader(x, batch_size=BATCH_SIZE, shuffle=False)
    return run_epoch(model, loader, optimizer=None, beta=beta)


def _checkpoint_payload(
    model: nn.Module,
    history: dict,
    *,
    epoch: int,
    best_epoch: int | None,
    best_val_total: float | None,
    best_val_token_acc: float | None,
    epochs_no_improve: int,
    optimizer=None,
    scheduler=None,
    test_metrics: dict | None = None,
):
    return {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "epoch": int(epoch),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "best_val_total": float(best_val_total) if best_val_total is not None else None,
        "best_val_token_acc": float(best_val_token_acc) if best_val_token_acc is not None else None,
        "epochs_no_improve": int(epochs_no_improve),
        "token_to_idx": TOKEN_TO_IDX,
        "seq_len": SEQ_LEN,
        "vocab_size": VOCAB_SIZE,
        "max_len": MAX_LEN,
        "pad_idx": PAD_IDX,
        "unk_idx": UNK_IDX,
        "eos_idx": EOS_IDX,
        "history": history,
        "test_metrics": test_metrics,
        "encoder_layout": "onehot_channels_first_seqconv",
        "decoder_output": "logits",
        "loss_name": "token_cross_entropy_plus_kl",
        "selection_metric": "val_token_acc",
    }


def _save_training_checkpoints(
    model: nn.Module,
    history: dict,
    *,
    save_dir: Path,
    checkpoint_stem: str,
    epoch: int,
    best_epoch: int | None,
    best_val_total: float | None,
    best_val_token_acc: float | None,
    epochs_no_improve: int,
    optimizer=None,
    scheduler=None,
    is_best: bool,
    save_epoch_checkpoints: bool,
):
    payload = _checkpoint_payload(
        model,
        history,
        epoch=epoch,
        best_epoch=best_epoch,
        best_val_total=best_val_total,
        best_val_token_acc=best_val_token_acc,
        epochs_no_improve=epochs_no_improve,
        optimizer=optimizer,
        scheduler=scheduler,
        test_metrics=None,
    )

    last_path = save_dir / f"{checkpoint_stem}_last.pt"
    torch.save(payload, last_path)

    if save_epoch_checkpoints:
        epoch_path = save_dir / f"{checkpoint_stem}_epoch_{epoch:03d}.pt"
        torch.save(payload, epoch_path)

    if is_best:
        best_path = save_dir / f"{checkpoint_stem}_best.pt"
        torch.save(payload, best_path)


def train_model(
    train_x: np.ndarray,
    val_x: np.ndarray,
    *,
    model: nn.Module | None = None,
    optimizer=None,
    scheduler=None,
    history: dict | None = None,
    start_epoch: int = 0,
    min_epochs: int = MIN_EPOCHS,
    max_epochs: int = MAX_EPOCHS,
    early_stopping_patience: int = EARLY_STOPPING_PATIENCE,
    checkpoint_dir: Path | None = None,
    checkpoint_stem: str | None = None,
    save_epoch_checkpoints: bool = SAVE_EPOCH_CHECKPOINTS,
    wandb_run_name: str | None = None,
    best_epoch: int | None = None,
    best_val_token_acc: float | None = None,
    epochs_no_improve: int = 0,
):
    if model is None:
        model = PaperLikeSelfiesVAE(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, latent_dim=LATENT_DIM).to(device)
    if optimizer is None:
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    if scheduler is None:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=LR_SCHEDULER_FACTOR,
            patience=LR_SCHEDULER_PATIENCE,
            min_lr=1e-6,
        )

    train_loader = make_loader(train_x, batch_size=BATCH_SIZE, shuffle=True)

    if history is None:
        history = {
            "beta": [],
            "lr": [],
            "train_total": [],
            "val_total": [],
            "train_recon_per_token": [],
            "val_recon_per_token": [],
            "train_kl": [],
            "val_kl": [],
            "train_token_acc": [],
            "val_token_acc": [],
        }
    else:
        for key in [
            "beta",
            "lr",
            "train_total",
            "val_total",
            "train_recon_per_token",
            "val_recon_per_token",
            "train_kl",
            "val_kl",
            "train_token_acc",
            "val_token_acc",
        ]:
            history.setdefault(key, [])

    if best_val_token_acc is None:
        if history["val_token_acc"]:
            best_idx = int(np.argmax(history["val_token_acc"]))
            best_epoch = best_idx + 1
            best_val_token_acc = float(history["val_token_acc"][best_idx])
        else:
            best_epoch = None
            best_val_token_acc = float("-inf")

    best_val_total = float(np.min(history["val_total"])) if history["val_total"] else float("inf")

    if checkpoint_dir is not None and checkpoint_stem is not None:
        checkpoint_dir = Path(checkpoint_dir)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

    wandb_run = init_wandb(run_name=wandb_run_name, epochs=max_epochs, start_epoch=start_epoch)

    early_stopped = False
    last_epoch = start_epoch

    if start_epoch >= max_epochs:
        print(f"start_epoch={start_epoch} already reached max_epochs={max_epochs}; skipping training loop.")

    for ep in range(start_epoch + 1, max_epochs + 1):
        beta = kl_beta(ep, KL_ANNEAL_EPOCHS)
        train_metrics = run_epoch(model, train_loader, optimizer=optimizer, beta=beta)
        val_metrics = evaluate(model, val_x, beta=beta)

        scheduler.step(val_metrics["token_acc"])
        current_lr = float(optimizer.param_groups[0]["lr"])

        history["beta"].append(beta)
        history["lr"].append(current_lr)
        history["train_total"].append(train_metrics["total"])
        history["val_total"].append(val_metrics["total"])
        history["train_recon_per_token"].append(train_metrics["recon_per_token"])
        history["val_recon_per_token"].append(val_metrics["recon_per_token"])
        history["train_kl"].append(train_metrics["kl"])
        history["val_kl"].append(val_metrics["kl"])
        history["train_token_acc"].append(train_metrics["token_acc"])
        history["val_token_acc"].append(val_metrics["token_acc"])

        if val_metrics["total"] < best_val_total:
            best_val_total = float(val_metrics["total"])

        is_best = val_metrics["token_acc"] > (best_val_token_acc + 1e-12)
        if is_best:
            best_val_token_acc = float(val_metrics["token_acc"])
            best_epoch = ep
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        print(
            f"epoch {ep:03d} | beta={beta:.2f} | lr={current_lr:.6f} | "
            f"train total={train_metrics['total']:.4f} | val total={val_metrics['total']:.4f} | "
            f"train recon/tok={train_metrics['recon_per_token']:.4f} | val recon/tok={val_metrics['recon_per_token']:.4f} | "
            f"train acc={train_metrics['token_acc']:.4f} | val acc={val_metrics['token_acc']:.4f}"
            + (" | new best val acc" if is_best else "")
        )

        if checkpoint_dir is not None and checkpoint_stem is not None:
            _save_training_checkpoints(
                model,
                history,
                save_dir=checkpoint_dir,
                checkpoint_stem=checkpoint_stem,
                epoch=ep,
                best_epoch=best_epoch,
                best_val_total=best_val_total,
                best_val_token_acc=best_val_token_acc,
                epochs_no_improve=epochs_no_improve,
                optimizer=optimizer,
                scheduler=scheduler,
                is_best=is_best,
                save_epoch_checkpoints=save_epoch_checkpoints,
            )

        if wandb_run is not None:
            wandb_run.log(
                {
                    "epoch": ep,
                    "beta": beta,
                    "lr": current_lr,
                    "train/total": train_metrics["total"],
                    "val/total": val_metrics["total"],
                    "train/recon_per_token": train_metrics["recon_per_token"],
                    "val/recon_per_token": val_metrics["recon_per_token"],
                    "train/kl": train_metrics["kl"],
                    "val/kl": val_metrics["kl"],
                    "train/token_acc": train_metrics["token_acc"],
                    "val/token_acc": val_metrics["token_acc"],
                    "best/val_total": best_val_total,
                    "best/val_token_acc": best_val_token_acc,
                    "best/epoch": best_epoch,
                    "early_stop/epochs_no_improve": epochs_no_improve,
                },
                step=ep,
            )

        last_epoch = ep

        if ep >= min_epochs and epochs_no_improve >= early_stopping_patience:
            early_stopped = True
            print(
                f"Early stopping at epoch {ep}: no val token-accuracy improvement for "
                f"{epochs_no_improve} epochs (patience={early_stopping_patience})."
            )
            break

    if wandb_run is not None:
        wandb_run.finish()

    checkpoint_info = {
        "best_epoch": best_epoch,
        "best_val_total": best_val_total,
        "best_val_token_acc": best_val_token_acc,
        "last_epoch": last_epoch,
        "epochs_no_improve": epochs_no_improve,
        "early_stopped": early_stopped,
    }
    return model, optimizer, scheduler, history, checkpoint_info

### Train the full pretraining model (ChemBL + Zinc + tox21 train, with tox21 val-augmented validation)

In [ ]:
save_dir = CHECKPOINT_DIR
save_dir.mkdir(parents=True, exist_ok=True)

print("Training on full pretraining data (ChemBL + Zinc + tox21_train) with tox21_val included in validation...")

last_ckpt_path = save_dir / f"{CHECKPOINT_STEM}_last.pt"

model_init = None
optimizer_init = None
scheduler_init = None
history_init = None
start_epoch = 0
best_epoch_init = None
best_val_token_acc_init = None
epochs_no_improve_init = 0

if AUTO_RESUME and last_ckpt_path.exists():
    print(f"Auto-resume enabled. Loading latest checkpoint: {last_ckpt_path}")
    resume_bundle = torch.load(last_ckpt_path, map_location=device)

    model_init = PaperLikeSelfiesVAE(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, latent_dim=LATENT_DIM).to(device)
    model_init.load_state_dict(resume_bundle["model_state_dict"])

    optimizer_init = torch.optim.Adam(model_init.parameters(), lr=LR)
    if resume_bundle.get("optimizer_state_dict") is not None:
        optimizer_init.load_state_dict(resume_bundle["optimizer_state_dict"])

    scheduler_init = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_init,
        mode="max",
        factor=LR_SCHEDULER_FACTOR,
        patience=LR_SCHEDULER_PATIENCE,
        min_lr=1e-6,
    )
    if resume_bundle.get("scheduler_state_dict") is not None:
        scheduler_init.load_state_dict(resume_bundle["scheduler_state_dict"])

    history_init = resume_bundle.get("history")
    if history_init is not None and "beta" in history_init:
        start_epoch = int(resume_bundle.get("epoch", len(history_init["beta"])))
    else:
        history_init = None
        start_epoch = int(resume_bundle.get("epoch", 0))

    best_epoch_init = resume_bundle.get("best_epoch")
    b = resume_bundle.get("best_val_token_acc")
    best_val_token_acc_init = float(b) if b is not None else None
    epochs_no_improve_init = int(resume_bundle.get("epochs_no_improve", 0))

    print(
        f"Resuming from epoch {start_epoch} with best_epoch={best_epoch_init} "
        f"and best_val_token_acc={best_val_token_acc_init}."
    )
elif AUTO_RESUME:
    print("Auto-resume enabled but no previous checkpoint found. Starting fresh.")
else:
    print("Auto-resume disabled. Starting fresh.")

model, optimizer, scheduler, history, checkpoint_info = train_model(
    train_x,
    val_x,
    model=model_init,
    optimizer=optimizer_init,
    scheduler=scheduler_init,
    history=history_init,
    start_epoch=start_epoch,
    min_epochs=MIN_EPOCHS,
    max_epochs=MAX_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    checkpoint_dir=save_dir,
    checkpoint_stem=CHECKPOINT_STEM,
    save_epoch_checkpoints=SAVE_EPOCH_CHECKPOINTS,
    best_epoch=best_epoch_init,
    best_val_token_acc=best_val_token_acc_init,
    epochs_no_improve=epochs_no_improve_init,
)

final_beta = history["beta"][-1] if history["beta"] else 1.0
test_metrics = evaluate(model, test_x, beta=final_beta)

# Keep a legacy single-checkpoint path for backwards compatibility.
ckpt_path = save_dir / f"{CHECKPOINT_STEM}.pt"
torch.save(
    _checkpoint_payload(
        model,
        history,
        epoch=checkpoint_info["last_epoch"],
        best_epoch=checkpoint_info["best_epoch"],
        best_val_total=checkpoint_info["best_val_total"],
        best_val_token_acc=checkpoint_info["best_val_token_acc"],
        epochs_no_improve=checkpoint_info["epochs_no_improve"],
        optimizer=optimizer,
        scheduler=scheduler,
        test_metrics=test_metrics,
    ),
    ckpt_path,
)

results_df = pd.DataFrame(
    [
        {
            "run_name": "full_chembl_zinc_tox21",
            "n_train": len(train_x),
            "n_val": len(val_x),
            "n_test": len(test_x),
            "last_epoch": checkpoint_info["last_epoch"],
            "early_stopped": checkpoint_info["early_stopped"],
            "final_train_total": history["train_total"][-1],
            "final_val_total": history["val_total"][-1],
            "final_train_token_acc": history["train_token_acc"][-1],
            "final_val_token_acc": history["val_token_acc"][-1],
            "best_epoch": checkpoint_info["best_epoch"],
            "best_val_token_acc": checkpoint_info["best_val_token_acc"],
            "best_val_total": checkpoint_info["best_val_total"],
            "test_total": test_metrics["total"],
            "test_recon_per_token": test_metrics["recon_per_token"],
            "test_kl": test_metrics["kl"],
            "test_token_acc": test_metrics["token_acc"],
            "checkpoint_best": str(save_dir / f"{CHECKPOINT_STEM}_best.pt"),
            "checkpoint_last": str(save_dir / f"{CHECKPOINT_STEM}_last.pt"),
            "checkpoint_compat": str(ckpt_path),
        }
    ]
)
results_df

### Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = np.arange(1, len(history["train_total"]) + 1)

axes[0].plot(epochs, history["train_total"], label="train")
axes[0].plot(epochs, history["val_total"], label="val")
axes[0].set_title("Total loss / sample")
axes[0].set_xlabel("epoch")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(epochs, history["train_recon_per_token"], label="train")
axes[1].plot(epochs, history["val_recon_per_token"], label="val")
axes[1].set_title("Reconstruction CE / token")
axes[1].set_xlabel("epoch")
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, history["train_token_acc"], label="train")
axes[2].plot(epochs, history["val_token_acc"], label="val")
axes[2].set_title("Token accuracy")
axes[2].set_xlabel("epoch")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Quick reconstruction check

In [ ]:
def decode_ids_to_selfies(ids: np.ndarray | list[int]) -> str:
    toks = []
    for idx in ids:
        tok = IDX_TO_TOKEN[int(idx)]
        if tok == EOS:
            break
        if tok == PAD:
            continue
        toks.append(tok)
    return "".join(toks)


def show_reconstructions(n: int = 5, seed: int = 42):
    model.eval()

    rng = np.random.default_rng(seed)
    k = min(n, len(test_x))
    idxs = rng.choice(len(test_x), size=k, replace=False)

    x_ids = torch.from_numpy(test_x[idxs]).long().to(device)
    x_onehot = ids_to_onehot(x_ids, VOCAB_SIZE)

    with torch.no_grad():
        logits, _, _ = model(x_onehot)
        pred_ids = logits.argmax(dim=-1).cpu().numpy()

    for j, idx in enumerate(idxs):
        orig_selfies = test_selfies[idx]
        pred_selfies = decode_ids_to_selfies(pred_ids[j])
        exact = orig_selfies == pred_selfies
        print(f"[{j}] exact={exact}")
        print("orig:", orig_selfies)
        print("pred:", pred_selfies)
        print()


show_reconstructions(n=5, seed=SEED)


### Notes for downstream tox21 classification
- The best checkpoint is selected using **validation token accuracy** (`*_best.pt`).
- The latest checkpoint (`*_last.pt`) is updated every epoch and contains optimizer/scheduler/history state for crash recovery.

### Crash-resume behavior
With `AUTO_RESUME=True`, the main training cell above will resume automatically from `*_last.pt` if present.

# Tox21 downstream evaluation

## XGBoost on tox21 latent features (train on tox21 train, report val/test AUROC + AUPRC)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    precision_recall_curve,
)

TOX21_TRAIN_PATH = Path("data") / "Train" / "tox21_train_clean.csv"
TOX21_VAL_PATH = Path("data") / "Val" / "tox21_val_clean.csv"
TOX21_TEST_PATH = Path("data") / "Test" / "tox21_test_clean.csv"

best_ckpt_path = CHECKPOINT_DIR / f"{CHECKPOINT_STEM}_best.pt"
if not best_ckpt_path.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {best_ckpt_path}")

bundle = torch.load(best_ckpt_path, map_location=device)
model_best = PaperLikeSelfiesVAE(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, latent_dim=LATENT_DIM).to(device)
model_best.load_state_dict(bundle["model_state_dict"])
model_best.eval()

TASKS = [c for c in pd.read_csv(TOX21_TRAIN_PATH, nrows=1).columns if c.startswith("NR-") or c.startswith("SR-")]


def _smiles_to_ids_for_vocab(smiles: str):
    try:
        s = sf.encoder(smiles)
        toks = list(sf.split_selfies(s))
    except Exception:
        return None

    if len(toks) > MAX_LEN:
        return None

    ids = [TOKEN_TO_IDX.get(tok, UNK_IDX) for tok in toks[:MAX_LEN]]
    ids.append(EOS_IDX)

    arr = np.full(SEQ_LEN, PAD_IDX, dtype=np.int64)
    arr[:len(ids)] = ids
    return arr


def build_latent_split(csv_path: Path, task_cols: list[str], batch_size: int = 512):
    df = pd.read_csv(csv_path).dropna(subset=["canonical_smiles"]).reset_index(drop=True)
    y_all = df[task_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

    x_ids_list, y_kept = [], []
    failed = 0
    for smi, y in zip(df["canonical_smiles"].astype(str).tolist(), y_all):
        ids = _smiles_to_ids_for_vocab(smi)
        if ids is None:
            failed += 1
            continue
        x_ids_list.append(ids)
        y_kept.append(y)

    x_ids = np.stack(x_ids_list)
    y = np.stack(y_kept)

    z_chunks = []
    with torch.no_grad():
        for i in range(0, len(x_ids), batch_size):
            xb = torch.from_numpy(x_ids[i:i + batch_size]).long().to(device)
            x_onehot_seq_vocab = F.one_hot(xb, num_classes=VOCAB_SIZE).float()
            x_onehot_vocab_seq = x_onehot_seq_vocab.transpose(1, 2).contiguous()
            mean, _ = model_best.encoder(x_onehot_vocab_seq)
            z_chunks.append(mean.cpu().numpy())

    z = np.vstack(z_chunks)
    return z, y, failed


Z_train, Y_train, fail_train = build_latent_split(TOX21_TRAIN_PATH, TASKS)
Z_val, Y_val, fail_val = build_latent_split(TOX21_VAL_PATH, TASKS)
Z_test, Y_test, fail_test = build_latent_split(TOX21_TEST_PATH, TASKS)

print("Loaded latent feature matrices:")
print(f"train: Z={Z_train.shape}, Y={Y_train.shape}, skipped={fail_train}")
print(f"val  : Z={Z_val.shape}, Y={Y_val.shape}, skipped={fail_val}")
print(f"test : Z={Z_test.shape}, Y={Y_test.shape}, skipped={fail_test}")

In [ ]:
def _binary_metrics(y_true: np.ndarray, probs: np.ndarray, threshold: float) -> dict:
    y_true = y_true.astype(int)
    y_hat = (probs >= threshold).astype(int)

    out = {
        "n": int(len(y_true)),
        "Accuracy": float(accuracy_score(y_true, y_hat)) if len(y_true) else np.nan,
        "F1": float(f1_score(y_true, y_hat, zero_division=0)) if len(y_true) else np.nan,
        "Precision": float(precision_score(y_true, y_hat, zero_division=0)) if len(y_true) else np.nan,
        "Recall": float(recall_score(y_true, y_hat, zero_division=0)) if len(y_true) else np.nan,
        "AUROC": np.nan,
        "AUPRC": np.nan,
    }

    if len(y_true) and len(np.unique(y_true)) >= 2:
        out["AUROC"] = float(roc_auc_score(y_true, probs))
        out["AUPRC"] = float(average_precision_score(y_true, probs))

    return out


task_rows = []
roc_curves = {}
pr_curves = {}

for j, task in enumerate(TASKS):
    ytr, yv, yt = Y_train[:, j], Y_val[:, j], Y_test[:, j]

    mtr = np.isin(ytr, [0, 1])
    mv = np.isin(yv, [0, 1])
    mt = np.isin(yt, [0, 1])

    # Need both classes for training.
    if len(np.unique(ytr[mtr])) < 2:
        continue

    clf = XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1,
    )

    fit_kwargs = {"verbose": False}
    if int(mv.sum()) > 0:
        fit_kwargs["eval_set"] = [(Z_val[mv], yv[mv])]

    clf.fit(Z_train[mtr], ytr[mtr], **fit_kwargs)

    if int(mv.sum()) > 0:
        p_val = clf.predict_proba(Z_val[mv])[:, 1]
        y_val = yv[mv].astype(int)
    else:
        p_val = np.array([])
        y_val = np.array([], dtype=int)

    if len(y_val) and len(np.unique(y_val)) >= 2:
        thresholds = np.linspace(0.05, 0.95, 19)
        best_t = max(
            thresholds,
            key=lambda t: f1_score(y_val, (p_val >= t).astype(int), zero_division=0),
        )
    else:
        best_t = 0.5

    p_test = clf.predict_proba(Z_test[mt])[:, 1] if int(mt.sum()) > 0 else np.array([])
    y_test = yt[mt].astype(int) if int(mt.sum()) > 0 else np.array([], dtype=int)

    val_res = _binary_metrics(y_val, p_val, best_t) if len(y_val) else {k: np.nan for k in ["n", "AUROC", "AUPRC", "Accuracy", "F1", "Precision", "Recall"]}
    test_res = _binary_metrics(y_test, p_test, best_t) if len(y_test) else {k: np.nan for k in ["n", "AUROC", "AUPRC", "Accuracy", "F1", "Precision", "Recall"]}

    task_rows.append(
        {
            "task": task,
            "threshold": float(best_t),
            "n_val": int(val_res["n"]) if not np.isnan(val_res["n"]) else 0,
            "Val_AUROC": val_res["AUROC"],
            "Val_AUPRC": val_res["AUPRC"],
            "Val_Accuracy": val_res["Accuracy"],
            "Val_F1": val_res["F1"],
            "Val_Precision": val_res["Precision"],
            "Val_Recall": val_res["Recall"],
            "n_test": int(test_res["n"]) if not np.isnan(test_res["n"]) else 0,
            "Test_AUROC": test_res["AUROC"],
            "Test_AUPRC": test_res["AUPRC"],
            "Test_Accuracy": test_res["Accuracy"],
            "Test_F1": test_res["F1"],
            "Test_Precision": test_res["Precision"],
            "Test_Recall": test_res["Recall"],
        }
    )

    # Keep plotting dictionaries on the tox21 test split for comparability.
    if len(y_test) and len(np.unique(y_test)) >= 2:
        fpr, tpr, _ = roc_curve(y_test, p_test)
        prec, rec, _ = precision_recall_curve(y_test, p_test)
        roc_curves[task] = (fpr, tpr, float(test_res["AUROC"]))
        pr_curves[task] = (rec, prec, float(test_res["AUPRC"]), float(y_test.mean()))

if task_rows:
    metrics_df = pd.DataFrame(task_rows).sort_values("Test_AUROC", ascending=False).reset_index(drop=True)
else:
    metrics_df = pd.DataFrame(
        columns=[
            "task",
            "threshold",
            "n_val",
            "Val_AUROC",
            "Val_AUPRC",
            "Val_Accuracy",
            "Val_F1",
            "Val_Precision",
            "Val_Recall",
            "n_test",
            "Test_AUROC",
            "Test_AUPRC",
            "Test_Accuracy",
            "Test_F1",
            "Test_Precision",
            "Test_Recall",
        ]
    )
metrics_df

In [ ]:
if metrics_df.empty:
    print("No tox21 tasks had sufficient class variation for train/val/test evaluation.")
else:
    print("Macro average metrics (validation + test):")
    metrics_df[
        [
            "Val_AUROC",
            "Val_AUPRC",
            "Val_Accuracy",
            "Test_AUROC",
            "Test_AUPRC",
            "Test_Accuracy",
        ]
    ].mean(numeric_only=True).to_frame("macro_mean").T

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for ax, task in zip(axes.ravel(), TASKS):
    if task in roc_curves:
        fpr, tpr, auc = roc_curves[task]
        ax.plot(fpr, tpr, label=f"AUROC={auc:.3f}")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
        ax.set_title(task)
        ax.set_xlabel("FPR")
        ax.set_ylabel("TPR")
        ax.legend(loc="lower right")
    else:
        ax.set_title(task)
        ax.text(0.5, 0.5, "insufficient class variation", ha="center", va="center")
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for ax, task in zip(axes.ravel(), TASKS):
    if task in pr_curves:
        rec, prec, ap, base = pr_curves[task]
        ax.plot(rec, prec, label=f"AUPRC={ap:.3f}")
        ax.hlines(base, 0, 1, colors="k", linestyles="--", alpha=0.4)
        ax.set_title(task)
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.legend(loc="lower left")
    else:
        ax.set_title(task)
        ax.text(0.5, 0.5, "insufficient class variation", ha="center", va="center")
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Tox21 test token accuracy using BEST pretrained checkpoint
TOX21_TEST_PATH = Path("data") / "Test" / "tox21_test_clean.csv"
best_ckpt_path = CHECKPOINT_DIR / f"{CHECKPOINT_STEM}_best.pt"

if not best_ckpt_path.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {best_ckpt_path}")

# 1) Load tox21 test SMILES
tox21_test_df = pd.read_csv(TOX21_TEST_PATH)
tox21_test_smiles = tox21_test_df["canonical_smiles"].dropna().astype(str).tolist()
tox21_test_smiles = list(dict.fromkeys(tox21_test_smiles))

# 2) Convert + filter exactly like pretraining pipeline
tox21_test_selfies, tox21_failed = smiles_to_selfies(tox21_test_smiles)
tox21_test_selfies = filter_selfies_len(tox21_test_selfies, max_len=MAX_LEN)
tox21_test_x = encode_list(tox21_test_selfies)

# 3) Load best model weights
bundle = torch.load(best_ckpt_path, map_location=device)
model_best = PaperLikeSelfiesVAE(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, latent_dim=LATENT_DIM).to(device)
model_best.load_state_dict(bundle["model_state_dict"])

# 4) Evaluate (same token-accuracy computation path as this notebook)
beta_eval = bundle["history"]["beta"][-1] if "history" in bundle and "beta" in bundle["history"] else 1.0
tox21_metrics = evaluate(model_best, tox21_test_x, beta=beta_eval)

print(f"Best checkpoint: {best_ckpt_path.name}")
print(f"Tox21 test SMILES: raw={len(tox21_test_smiles):,} | selfies_ok={len(tox21_test_selfies):,} | failed={tox21_failed:,}")
print(
    f"Tox21 test metrics -> "
    f"total={tox21_metrics['total']:.4f}, "
    f"recon_per_token={tox21_metrics['recon_per_token']:.4f}, "
    f"kl={tox21_metrics['kl']:.4f}, "
    f"token_acc={tox21_metrics['token_acc']:.4f}"
)